# Step 5 Fresh Run: Phase-1 from scratch + T4x2 parallel comparison

Use this notebook when no checkpoint remains in `/kaggle/working`.

It does **not edit source code**. It only:

1. pulls the public repo/branch,
2. prepares RegDB,
3. trains a fresh phase-1 checkpoint for 5 epochs,
4. launches two phase-2-only jobs in parallel on T4x2:
   - baseline, no UPR-CRE,
   - UPR-CRE v0.1 with `beta=0.1`, `gamma=0.5`, `warmup=2`,
5. collects `R1 / mAP / mINP` and relation diagnostics into CSV artifacts.

## Block 0 — Configuration

Change only this block if needed. The default RegDB source matches your Kaggle dataset mount.

In [ ]:
from pathlib import Path
import os, time

CFG = {
    # Public repository
    "REPO_URL": "https://github.com/TranTruongMMCII/UIT.CS2309.git",
    "REPO_NAME": "UIT.CS2309",
    "BRANCH": "feature/upr-cre",
    "WORK_DIR": "/kaggle/working",

    # Dataset
    "DATA_ROOT": "/kaggle/working/VIREID_Dataset",
    "REGDB_SOURCE": "/kaggle/input/datasets/xqq027/reg-db/RegDB",

    # Experiment settings
    "SEED": "1",
    "TRIAL": "1",
    "STAGE1_EPOCH": "5",
    "PHASE2_EPOCH": "5",
    "LR": "0.00045",
    "MILESTONES": "8 12",
    "BATCH_PIDNUM": "5",
    "PID_NUMSAMPLE": "4",
    "TEST_BATCH": "64",
    "NUM_WORKERS": "2",

    # UPR-CRE v0.1 mini-ablation config
    "UPR_BETA": "0.1",
    "UPR_GAMMA": "0.5",
    "UPR_MARGIN_WEIGHT": "1.0",
    "UPR_WARMUP_EPOCH": "2",

    # Run ID prefix
    "RUN_PREFIX": "step5fresh",

    # GitHub backup options. Requires Kaggle Secret named GITHUB_TOKEN.
    "ENABLE_GITHUB_BACKUP": True,
    "GITHUB_TOKEN_SECRET": "GITHUB_TOKEN",
    "GITHUB_OWNER": "TranTruongMMCII",
    "GITHUB_REPO": "UIT.CS2309",
    "GIT_USER_NAME": "Kaggle Bot",
    "GIT_USER_EMAIL": "kaggle-bot@example.com",
}

run_stamp = time.strftime("%Y%m%d_%H%M%S")
CFG["RUN_STAMP"] = run_stamp

base_env = Path(CFG["WORK_DIR"]) / "step5_base_env.sh"
base_env.write_text("\n".join([f"export {k}='{v}'" for k, v in CFG.items()]) + "\n")
print(base_env)
print("Config:")
for k, v in CFG.items():
    print(f"  {k}={v}")

## Block 1 — Pull repo and checkout branch

This block resets the Kaggle copy to `origin/feature/upr-cre`. It does not change code.

In [4]:
%%bash
set -euo pipefail
source /kaggle/working/step5_base_env.sh
cd "$WORK_DIR"

if [ ! -d "$REPO_NAME/.git" ]; then
  git clone --branch "$BRANCH" "$REPO_URL" "$REPO_NAME"
else
  cd "$REPO_NAME"
  git fetch origin "$BRANCH"
  git checkout "$BRANCH"
  git reset --hard "origin/$BRANCH"
  cd "$WORK_DIR"
fi

cd "$WORK_DIR/$REPO_NAME"
COMMIT=$(git rev-parse --short HEAD)
echo "export COMMIT='$COMMIT'" > /kaggle/working/step5_runtime_env.sh
echo "export RUN_SUFFIX='${RUN_PREFIX}_${RUN_STAMP}_${COMMIT}'" >> /kaggle/working/step5_runtime_env.sh

echo "Repo: $(pwd)"
echo "Branch: $(git branch --show-current)"
echo "Commit: $COMMIT"
echo "Status:"
git status --short

Your branch is up to date with 'origin/feature/upr-cre'.
HEAD is now at 64f0adf Change output flag to csv-output in script
Repo: /kaggle/working/UIT.CS2309
Branch: feature/upr-cre
Commit: 64f0adf
Status:


From https://github.com/TranTruongMMCII/UIT.CS2309
 * branch            feature/upr-cre -> FETCH_HEAD
Already on 'feature/upr-cre'


## Block 2 — Install Kaggle requirements and prepare RegDB

This uses the scripts already committed in your repo. It also checks that two GPUs are visible.

In [5]:
%%bash
set -euo pipefail
source /kaggle/working/step5_base_env.sh
source /kaggle/working/step5_runtime_env.sh
cd "$WORK_DIR/$REPO_NAME/WSL_ReID"

python -m pip install -q -r requirements-kaggle.txt
python scripts/apply_kaggle_compat_patches.py
python scripts/prepare_regdb_kaggle.py --data-root "$DATA_ROOT" --regdb-source "$REGDB_SOURCE"
python scripts/check_kaggle_env.py --data-root "$DATA_ROOT"

python - <<'PY'
import torch
n = torch.cuda.device_count()
print("GPU count:", n)
for i in range(n):
    print(i, torch.cuda.get_device_name(i))
if n < 2:
    raise SystemExit("This notebook expects T4x2. Please enable GPU T4 x2 in Kaggle settings.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 770.1 kB/s eta 0:00:00
No compatibility patches were needed.
RegDB source: /kaggle/input/datasets/xqq027/reg-db/RegDB
RegDB prepared at: /kaggle/working/VIREID_Dataset/RegDB
Use training argument: --data-path /kaggle/working/VIREID_Dataset
=== PyTorch / GPU ===
torch: 2.10.0+cu128
cuda available: True
gpu count: 2
gpu 0: Tesla T4
gpu 1: Tesla T4

=== RegDB layout ===
/kaggle/working/VIREID_Dataset/RegDB/idx/train_visible_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/train_thermal_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/test_visible_1.txt OK
/kaggle/working/VIREID_Dataset/RegDB/idx/test_thermal_1.txt OK

 train_visible_1.txt
first line: Visible/285/male_back_v_05528_285.bmp 0
image: /kaggle/working/VIREID_Dataset/RegDB/Visible/285/male_back_v_05528_285.bmp
exists: True
image size: (64, 128) mode: RGB label: 0

 train_thermal_1.txt
first line: Thermal/285/male_back_t_05528_285.bmp 0
image: /kaggle/working/VIREID_Datas

bash: line 18: warning: here-document at line 11 delimited by end-of-file (wanted `PY')


## Block 3 — Static checks

This checks the code/scripts needed for Step 5. It does not check markdown docs.

In [6]:
%%bash
set -euo pipefail
source /kaggle/working/step5_base_env.sh
cd "$WORK_DIR/$REPO_NAME/WSL_ReID"

required=(
  main.py
  wsl.py
  task/train.py
  relation_metrics.py
  requirements-kaggle.txt
  scripts/apply_kaggle_compat_patches.py
  scripts/prepare_regdb_kaggle.py
  scripts/check_kaggle_env.py
  scripts/collect_relation_stats.py
)
for f in "${required[@]}"; do
  test -f "$f" || { echo "Missing $f"; exit 1; }
done

python -m py_compile main.py wsl.py task/train.py relation_metrics.py scripts/collect_relation_stats.py

grep -q -- "--upr-cre" main.py
grep -q -- "--save-relation-stats" main.py
grep -q -- "--csv-output" scripts/collect_relation_stats.py
grep -q -- "_apply_upr_cre_score" wsl.py

echo "Static checks OK."

Static checks OK.


## Block 4 — Train fresh phase-1 checkpoint, 5 epochs

This is the only non-parallel part. It trains phase 1 once, saves `model_5.pth`, and uses that checkpoint for both phase-2 jobs.

Expected time on T4: roughly similar to your previous phase-1 run.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/step5_base_env.sh
source /kaggle/working/step5_runtime_env.sh
cd "$WORK_DIR/$REPO_NAME/WSL_ReID"
mkdir -p /kaggle/working/run_logs

PHASE1_RUN="phase1_s${STAGE1_EPOCH}_${RUN_SUFFIX}"
echo "export PHASE1_RUN='$PHASE1_RUN'" >> /kaggle/working/step5_runtime_env.sh

echo "[Phase1] run=$PHASE1_RUN"
echo "[Phase1] commit=$COMMIT"

CUDA_VISIBLE_DEVICES=0 python main.py \
  --dataset regdb \
  --data-path "$DATA_ROOT" \
  --debug wsl \
  --save-path "$PHASE1_RUN" \
  --arch resnet \
  --trial "$TRIAL" \
  --seed "$SEED" \
  --stage1-epoch "$STAGE1_EPOCH" \
  --stage2-epoch 0 \
  --lr "$LR" \
  --batch-pidnum "$BATCH_PIDNUM" \
  --pid-numsample "$PID_NUMSAMPLE" \
  --test-batch "$TEST_BATCH" \
  --num-workers "$NUM_WORKERS" \
  --device 0 \
  --milestones $MILESTONES \
  2>&1 | tee "/kaggle/working/run_logs/${PHASE1_RUN}.log"

## Block 5 — Find and archive the phase-1 checkpoint

Run this after Block 4. It finds the `model_5.pth` created by the fresh phase-1 run.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/step5_base_env.sh
source /kaggle/working/step5_runtime_env.sh
cd "$WORK_DIR/$REPO_NAME/WSL_ReID"

PHASE1_CKPT=$(python - <<'PY'
import os
from pathlib import Path
phase1_run = os.environ["PHASE1_RUN"]
root = Path("/kaggle/working")
files = sorted(
    [p for p in root.rglob("model_5.pth") if phase1_run in str(p) and "saved_pretrain" in str(p)],
    key=lambda p: p.stat().st_mtime,
    reverse=True,
)
if files:
    print(files[0])
PY
)

if [ -z "$PHASE1_CKPT" ] || [ ! -f "$PHASE1_CKPT" ]; then
  echo "ERROR: could not find phase1 checkpoint for run $PHASE1_RUN"
  echo "All model_5.pth candidates:"
  find /kaggle/working -name model_5.pth | sort || true
  exit 1
fi

echo "export PHASE1_CKPT='$PHASE1_CKPT'" >> /kaggle/working/step5_runtime_env.sh
echo "PHASE1_CKPT=$PHASE1_CKPT"
ls -lh "$PHASE1_CKPT"

mkdir -p /kaggle/working/artifacts_step5_fresh/checkpoints
cp "$PHASE1_CKPT" /kaggle/working/artifacts_step5_fresh/checkpoints/phase1_model_5.pth
cp "/kaggle/working/run_logs/${PHASE1_RUN}.log" /kaggle/working/artifacts_step5_fresh/phase1_log.txt

## Block 5.5 — Backup phase-1 checkpoint to GitHub

Run this immediately after Block 5 if you want to survive a Kaggle session reset.

This block does two things:

1. commits **small metadata/log files** to `experiments/kaggle_runs/<RUN_SUFFIX>/`,
2. uploads the large `phase1_model_5.pth` checkpoint to a **GitHub Release asset**.

It does **not** commit `.pth` files to the git repository.

Requirements: Kaggle Secret `GITHUB_TOKEN` with repo write permission.


In [ ]:
from pathlib import Path
import os, json, hashlib, subprocess, textwrap, time, sys

try:
    from kaggle_secrets import UserSecretsClient
except Exception as exc:
    raise RuntimeError("Kaggle Secrets are required for GitHub backup. Add GITHUB_TOKEN in Add-ons -> Secrets.") from exc

try:
    import requests
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "requests"], check=True)
    import requests

cfg = CFG
if not cfg.get("ENABLE_GITHUB_BACKUP", False):
    print("GitHub backup disabled by CFG['ENABLE_GITHUB_BACKUP']=False")
else:
    work_dir = Path(cfg["WORK_DIR"])
    repo_dir = work_dir / cfg["REPO_NAME"]

    runtime = {}
    for line in (work_dir / "step5_runtime_env.sh").read_text().splitlines():
        if line.startswith("export ") and "=" in line:
            k, v = line[len("export "):].split("=", 1)
            runtime[k] = v.strip().strip("'").strip('"')

    run_suffix = runtime["RUN_SUFFIX"]
    phase1_run = runtime["PHASE1_RUN"]
    phase1_ckpt = Path(runtime["PHASE1_CKPT"])
    commit = runtime["COMMIT"]

    if not phase1_ckpt.exists():
        raise FileNotFoundError(f"Missing phase1 checkpoint: {phase1_ckpt}")

    token = UserSecretsClient().get_secret(cfg["GITHUB_TOKEN_SECRET"])
    owner = cfg["GITHUB_OWNER"]
    repo = cfg["GITHUB_REPO"]
    branch = cfg["BRANCH"]

    def sha256_file(path: Path, block_size: int = 1024 * 1024) -> str:
        h = hashlib.sha256()
        with path.open("rb") as f:
            for block in iter(lambda: f.read(block_size), b""):
                h.update(block)
        return h.hexdigest()

    ckpt_sha256 = sha256_file(phase1_ckpt)
    ckpt_size = phase1_ckpt.stat().st_size

    headers = {
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
        "X-GitHub-Api-Version": "2022-11-28",
    }

    tag = f"step5-{run_suffix}"
    release_name = f"Step 5 artifacts {run_suffix}"
    release_body = textwrap.dedent(f"""
    Step 5 fresh phase-1 checkpoint and run metadata.

    - Branch: {branch}
    - Commit: {commit}
    - Phase1 run: {phase1_run}
    - Checkpoint file: {phase1_ckpt.name}
    - Checkpoint size: {ckpt_size} bytes
    - Checkpoint sha256: {ckpt_sha256}
    """).strip()

    url = f"https://api.github.com/repos/{owner}/{repo}/releases/tags/{tag}"
    r = requests.get(url, headers=headers, timeout=60)
    if r.status_code == 404:
        create_url = f"https://api.github.com/repos/{owner}/{repo}/releases"
        payload = {
            "tag_name": tag,
            "target_commitish": branch,
            "name": release_name,
            "body": release_body,
            "draft": False,
            "prerelease": True,
        }
        r = requests.post(create_url, headers=headers, json=payload, timeout=60)
    if not r.ok:
        raise RuntimeError(f"GitHub release get/create failed: {r.status_code} {r.text[:500]}")
    release = r.json()
    release_url = release.get("html_url", "")

    asset_name = f"phase1_model_5_{run_suffix}.pth"
    for asset in release.get("assets", []):
        if asset.get("name") == asset_name:
            dr = requests.delete(asset["url"], headers=headers, timeout=60)
            if dr.status_code not in (204, 404):
                raise RuntimeError(f"Could not delete existing asset: {dr.status_code} {dr.text[:300]}")

    upload_url = release["upload_url"].split("{")[0]
    upload_headers = dict(headers)
    upload_headers["Content-Type"] = "application/octet-stream"
    with phase1_ckpt.open("rb") as f:
        ur = requests.post(
            f"{upload_url}?name={asset_name}",
            headers=upload_headers,
            data=f,
            timeout=600,
        )
    if not ur.ok:
        raise RuntimeError(f"Checkpoint upload failed: {ur.status_code} {ur.text[:500]}")
    asset = ur.json()
    asset_url = asset.get("browser_download_url", "")

    exp_dir = repo_dir / "experiments" / "kaggle_runs" / run_suffix
    exp_dir.mkdir(parents=True, exist_ok=True)

    manifest = {
        "run_suffix": run_suffix,
        "branch": branch,
        "commit": commit,
        "phase1_run": phase1_run,
        "phase1_checkpoint_release_asset": asset_url,
        "phase1_checkpoint_release_page": release_url,
        "phase1_checkpoint_name": asset_name,
        "phase1_checkpoint_size_bytes": ckpt_size,
        "phase1_checkpoint_sha256": ckpt_sha256,
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "config": cfg,
    }
    (exp_dir / "phase1_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    (exp_dir / "phase1_checkpoint_url.txt").write_text(asset_url + "\n", encoding="utf-8")
    (exp_dir / "release_url.txt").write_text(release_url + "\n", encoding="utf-8")

    phase1_log = work_dir / "run_logs" / f"{phase1_run}.log"
    if phase1_log.exists():
        (exp_dir / "phase1_log.txt").write_text(phase1_log.read_text(errors="replace"), encoding="utf-8")

    resume_script = """#!/usr/bin/env bash
set -euo pipefail
mkdir -p /kaggle/working/restored_checkpoints
curl -L -o /kaggle/working/restored_checkpoints/{asset_name} {asset_url}
echo \"Downloaded to /kaggle/working/restored_checkpoints/{asset_name}\"
python - <<'PYVERIFY'
import hashlib
from pathlib import Path
path = Path('/kaggle/working/restored_checkpoints/{asset_name}')
expected = '{ckpt_sha256}'
h = hashlib.sha256(path.read_bytes()).hexdigest()
print('sha256:', h)
assert h == expected, 'SHA256 mismatch'
PYVERIFY
""".format(asset_name=asset_name, asset_url=asset_url, ckpt_sha256=ckpt_sha256)
    (exp_dir / "download_phase1_checkpoint.sh").write_text(resume_script, encoding="utf-8")

    subprocess.run(["git", "config", "user.name", cfg["GIT_USER_NAME"]], cwd=repo_dir, check=True)
    subprocess.run(["git", "config", "user.email", cfg["GIT_USER_EMAIL"]], cwd=repo_dir, check=True)
    subprocess.run(["git", "add", str(exp_dir.relative_to(repo_dir))], cwd=repo_dir, check=True)

    status = subprocess.check_output(["git", "status", "--short"], cwd=repo_dir).decode().strip()
    if status:
        subprocess.run(["git", "commit", "-m", f"exp: backup phase1 checkpoint metadata {run_suffix}"], cwd=repo_dir, check=True)

        askpass = work_dir / "git_askpass_backup.sh"
        askpass.write_text(textwrap.dedent(f"""\
        #!/bin/sh
        case \"$1\" in
          *Username*) echo \"{owner}\" ;;
          *Password*) echo \"{token}\" ;;
          *) echo \"\" ;;
        esac
        """), encoding="utf-8")
        askpass.chmod(0o700)
        env = os.environ.copy()
        env["GIT_ASKPASS"] = str(askpass)
        env["GIT_TERMINAL_PROMPT"] = "0"
        subprocess.run(["git", "push", "origin", branch], cwd=repo_dir, check=True, env=env)
    else:
        print("No new small metadata files to commit.")

    print("GitHub backup complete.")
    print("Release:", release_url)
    print("Checkpoint asset:", asset_url)


## Block 6 — Launch two phase-2-only jobs on T4x2

GPU 0: baseline phase2-only, 5 epochs.  
GPU 1: UPR-CRE v0.1 phase2-only, `beta=0.1`, `gamma=0.5`, `warmup=2`, 5 epochs.

Both jobs use the same phase-1 checkpoint from Block 5.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/step5_base_env.sh
source /kaggle/working/step5_runtime_env.sh
cd "$WORK_DIR/$REPO_NAME/WSL_ReID"
mkdir -p /kaggle/working/run_logs

BASELINE_RUN="baseline_p2s${PHASE2_EPOCH}_${RUN_SUFFIX}"
UPR_RUN="upr_b${UPR_BETA//./}_g${UPR_GAMMA//./}_w${UPR_WARMUP_EPOCH}_p2s${PHASE2_EPOCH}_${RUN_SUFFIX}"

echo "export BASELINE_RUN='$BASELINE_RUN'" >> /kaggle/working/step5_runtime_env.sh
echo "export UPR_RUN='$UPR_RUN'" >> /kaggle/working/step5_runtime_env.sh

echo "BASELINE_RUN=$BASELINE_RUN"
echo "UPR_RUN=$UPR_RUN"
echo "PHASE1_CKPT=$PHASE1_CKPT"

CUDA_VISIBLE_DEVICES=0 nohup python main.py \
  --dataset regdb \
  --data-path "$DATA_ROOT" \
  --debug wsl \
  --save-path "$BASELINE_RUN" \
  --arch resnet \
  --trial "$TRIAL" \
  --seed "$SEED" \
  --model-path "$PHASE1_CKPT" \
  --stage1-epoch 0 \
  --stage2-epoch "$PHASE2_EPOCH" \
  --lr "$LR" \
  --batch-pidnum "$BATCH_PIDNUM" \
  --pid-numsample "$PID_NUMSAMPLE" \
  --test-batch "$TEST_BATCH" \
  --num-workers "$NUM_WORKERS" \
  --device 0 \
  --milestones $MILESTONES \
  --save-relation-stats \
  --relation-stats-dir "../saved_regdb_resnet/${BASELINE_RUN}_${TRIAL}/relation_stats" \
  > "/kaggle/working/run_logs/${BASELINE_RUN}.log" 2>&1 &
PID_BASE=$!

CUDA_VISIBLE_DEVICES=1 nohup python main.py \
  --dataset regdb \
  --data-path "$DATA_ROOT" \
  --debug wsl \
  --save-path "$UPR_RUN" \
  --arch resnet \
  --trial "$TRIAL" \
  --seed "$SEED" \
  --model-path "$PHASE1_CKPT" \
  --stage1-epoch 0 \
  --stage2-epoch "$PHASE2_EPOCH" \
  --lr "$LR" \
  --batch-pidnum "$BATCH_PIDNUM" \
  --pid-numsample "$PID_NUMSAMPLE" \
  --test-batch "$TEST_BATCH" \
  --num-workers "$NUM_WORKERS" \
  --device 0 \
  --milestones $MILESTONES \
  --upr-cre \
  --upr-beta "$UPR_BETA" \
  --upr-gamma "$UPR_GAMMA" \
  --upr-margin-weight "$UPR_MARGIN_WEIGHT" \
  --upr-warmup-epoch "$UPR_WARMUP_EPOCH" \
  --save-relation-stats \
  --relation-stats-dir "../saved_regdb_resnet/${UPR_RUN}_${TRIAL}/relation_stats" \
  > "/kaggle/working/run_logs/${UPR_RUN}.log" 2>&1 &
PID_UPR=$!

cat > /kaggle/working/step5_parallel_pids.txt <<EOF
$PID_BASE $BASELINE_RUN
$PID_UPR $UPR_RUN
EOF

echo "Started jobs:"
cat /kaggle/working/step5_parallel_pids.txt

## Block 7 — Monitor jobs

Run this cell repeatedly while the two jobs are running.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/step5_base_env.sh
source /kaggle/working/step5_runtime_env.sh

echo "=== nvidia-smi ==="
nvidia-smi

echo "=== processes ==="
ps -ef | grep "python main.py" | grep -v grep || true

echo "=== baseline tail ==="
tail -40 "/kaggle/working/run_logs/${BASELINE_RUN}.log" || true

echo "=== UPR tail ==="
tail -40 "/kaggle/working/run_logs/${UPR_RUN}.log" || true

## Block 8 — Wait until both jobs finish

This polls the two background PIDs from Block 6. You can stop this cell if you only want manual monitoring.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/step5_base_env.sh
source /kaggle/working/step5_runtime_env.sh

if [ ! -f /kaggle/working/step5_parallel_pids.txt ]; then
  echo "Missing /kaggle/working/step5_parallel_pids.txt. Run Block 6 first."
  exit 1
fi

while true; do
  alive=0
  while read -r pid name; do
    if kill -0 "$pid" 2>/dev/null; then
      alive=1
    fi
  done < /kaggle/working/step5_parallel_pids.txt

  date
  nvidia-smi --query-gpu=index,name,utilization.gpu,memory.used,memory.total --format=csv,noheader || true
  tail -5 "/kaggle/working/run_logs/${BASELINE_RUN}.log" || true
  tail -5 "/kaggle/working/run_logs/${UPR_RUN}.log" || true

  if [ "$alive" -eq 0 ]; then
    echo "Both jobs finished."
    break
  fi
  sleep 60
done

## Block 9 — Collect metrics and relation summaries

This creates `/kaggle/working/step5_fresh_t4x2_summary.csv`.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/step5_base_env.sh
source /kaggle/working/step5_runtime_env.sh
cd "$WORK_DIR/$REPO_NAME/WSL_ReID"

for RUN in "${BASELINE_RUN}_${TRIAL}" "${UPR_RUN}_${TRIAL}"; do
  STATS_DIR="../saved_regdb_resnet/${RUN}/relation_stats"
  echo "Collecting relation stats for $RUN"
  if [ -d "$STATS_DIR" ]; then
    python scripts/collect_relation_stats.py \
      --stats-dir "$STATS_DIR" \
      --csv-output "$STATS_DIR/relation_stats_summary.csv"
  else
    echo "WARNING: missing stats dir $STATS_DIR"
  fi
  echo "==== $RUN log tail ===="
  grep -E "R1:|Best_R1|Epoch:" "../saved_regdb_resnet/${RUN}/log/log.txt" | tail -50 || true
  echo
done

python - <<'PY'
import csv
import os
import re
from pathlib import Path

trial = os.environ["TRIAL"]
base = Path("../saved_regdb_resnet")
runs = [os.environ["BASELINE_RUN"] + f"_{trial}", os.environ["UPR_RUN"] + f"_{trial}"]

rows = []
for run in runs:
    log_path = base / run / "log" / "log.txt"
    text = log_path.read_text(encoding="utf-8", errors="ignore") if log_path.exists() else ""
    metrics = re.findall(
        r"R1:([0-9.]+);\s+R10:([0-9.]+);\s+R20:([0-9.]+);\s+mAP:([0-9.]+);\s+mINP:([0-9.]+)",
        text,
    )
    if metrics:
        best_by_map = max(metrics, key=lambda x: float(x[3]))
        last = metrics[-1]
    else:
        best_by_map = ("", "", "", "", "")
        last = ("", "", "", "", "")

    stats_path = base / run / "relation_stats" / "relation_stats_summary.csv"
    last_stats = {}
    if stats_path.exists():
        with stats_path.open(newline="", encoding="utf-8") as f:
            stat_rows = list(csv.DictReader(f))
        if stat_rows:
            last_stats = stat_rows[-1]

    rows.append({
        "run": run,
        "best_r1_by_map": best_by_map[0],
        "best_map": best_by_map[3],
        "best_minp_by_map": best_by_map[4],
        "last_r1": last[0],
        "last_map": last[3],
        "last_minp": last[4],
        "last_common_accuracy": last_stats.get("common_accuracy", ""),
        "last_mutual_match_ratio": last_stats.get("mutual_match_ratio", ""),
        "last_num_common": last_stats.get("num_common", ""),
        "last_num_specific": last_stats.get("num_specific", ""),
        "last_num_remain": last_stats.get("num_remain", ""),
    })

out = Path("/kaggle/working/step5_fresh_t4x2_summary.csv")
with out.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print(out)
for row in rows:
    print(row)
PY

cat /kaggle/working/step5_fresh_t4x2_summary.csv

## Block 10 — Archive artifacts

Creates `/kaggle/working/artifacts_step5_fresh_t4x2.tar.gz` containing logs, summary CSV, relation summaries, and the phase-1 checkpoint copy.

In [ ]:
%%bash
set -euo pipefail
source /kaggle/working/step5_base_env.sh
source /kaggle/working/step5_runtime_env.sh
cd "$WORK_DIR/$REPO_NAME/WSL_ReID"

ART=/kaggle/working/artifacts_step5_fresh_t4x2
mkdir -p "$ART/logs" "$ART/relation_stats" "$ART/checkpoints"

cp /kaggle/working/step5_fresh_t4x2_summary.csv "$ART/" 2>/dev/null || true
cp /kaggle/working/run_logs/${PHASE1_RUN}.log "$ART/logs/" 2>/dev/null || true
cp /kaggle/working/run_logs/${BASELINE_RUN}.log "$ART/logs/" 2>/dev/null || true
cp /kaggle/working/run_logs/${UPR_RUN}.log "$ART/logs/" 2>/dev/null || true
cp /kaggle/working/artifacts_step5_fresh/checkpoints/phase1_model_5.pth "$ART/checkpoints/" 2>/dev/null || true

for RUN in "${BASELINE_RUN}_${TRIAL}" "${UPR_RUN}_${TRIAL}"; do
  if [ -f "../saved_regdb_resnet/${RUN}/relation_stats/relation_stats_summary.csv" ]; then
    cp "../saved_regdb_resnet/${RUN}/relation_stats/relation_stats_summary.csv" \
       "$ART/relation_stats/${RUN}_relation_stats_summary.csv"
  fi
  if [ -f "../saved_regdb_resnet/${RUN}/log/log.txt" ]; then
    cp "../saved_regdb_resnet/${RUN}/log/log.txt" "$ART/logs/${RUN}_log.txt"
  fi
done

tar -czf /kaggle/working/artifacts_step5_fresh_t4x2.tar.gz -C /kaggle/working artifacts_step5_fresh_t4x2
ls -lh /kaggle/working/artifacts_step5_fresh_t4x2.tar.gz
find "$ART" -maxdepth 3 -type f | sort

## Block 10.5 — Push final small artifacts to GitHub

Run this after Block 10. It commits small artifacts to `experiments/kaggle_runs/<RUN_SUFFIX>/`:

- final summary CSV,
- phase1/baseline/UPR logs,
- relation stats summaries,
- artifact manifest.

It does **not** commit `.pth` checkpoint files. The phase-1 checkpoint should already be in the GitHub Release from Block 5.5.


In [ ]:
from pathlib import Path
import os, json, shutil, subprocess, textwrap, time

try:
    from kaggle_secrets import UserSecretsClient
except Exception as exc:
    raise RuntimeError("Kaggle Secrets are required for GitHub backup. Add GITHUB_TOKEN in Add-ons -> Secrets.") from exc

cfg = CFG
if not cfg.get("ENABLE_GITHUB_BACKUP", False):
    print("GitHub backup disabled by CFG['ENABLE_GITHUB_BACKUP']=False")
else:
    work_dir = Path(cfg["WORK_DIR"])
    repo_dir = work_dir / cfg["REPO_NAME"]
    wsl_dir = repo_dir / "WSL_ReID"

    runtime = {}
    for line in (work_dir / "step5_runtime_env.sh").read_text().splitlines():
        if line.startswith("export ") and "=" in line:
            k, v = line[len("export "):].split("=", 1)
            runtime[k] = v.strip().strip("'").strip('"')

    run_suffix = runtime["RUN_SUFFIX"]
    phase1_run = runtime.get("PHASE1_RUN", "")
    baseline_run = runtime.get("BASELINE_RUN", "")
    upr_run = runtime.get("UPR_RUN", "")
    trial = cfg["TRIAL"]
    branch = cfg["BRANCH"]
    owner = cfg["GITHUB_OWNER"]
    token = UserSecretsClient().get_secret(cfg["GITHUB_TOKEN_SECRET"])

    exp_dir = repo_dir / "experiments" / "kaggle_runs" / run_suffix
    exp_dir.mkdir(parents=True, exist_ok=True)

    def copy_if_exists(src, dst):
        src = Path(src)
        dst = Path(dst)
        if src.exists():
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            return True
        return False

    copy_if_exists(work_dir / "step5_fresh_t4x2_summary.csv", exp_dir / "step5_fresh_t4x2_summary.csv")

    log_dir = exp_dir / "logs"
    copy_if_exists(work_dir / "run_logs" / f"{phase1_run}.log", log_dir / f"{phase1_run}.log")
    copy_if_exists(work_dir / "run_logs" / f"{baseline_run}.log", log_dir / f"{baseline_run}.log")
    copy_if_exists(work_dir / "run_logs" / f"{upr_run}.log", log_dir / f"{upr_run}.log")

    rel_dir = exp_dir / "relation_stats"
    saved_root = wsl_dir / ".." / "saved_regdb_resnet"
    for run in [f"{baseline_run}_{trial}", f"{upr_run}_{trial}"]:
        copy_if_exists(saved_root / run / "relation_stats" / "relation_stats_summary.csv", rel_dir / f"{run}_relation_stats_summary.csv")
        copy_if_exists(saved_root / run / "log" / "log.txt", log_dir / f"{run}_log.txt")

    manifest = {
        "run_suffix": run_suffix,
        "branch": branch,
        "commit_after_backup": subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=repo_dir).decode().strip(),
        "phase1_run": phase1_run,
        "baseline_run": baseline_run,
        "upr_run": upr_run,
        "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
        "note": "Small artifacts only. Checkpoint is stored as GitHub Release asset from Block 5.5.",
    }
    (exp_dir / "final_artifact_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")

    subprocess.run(["git", "config", "user.name", cfg["GIT_USER_NAME"]], cwd=repo_dir, check=True)
    subprocess.run(["git", "config", "user.email", cfg["GIT_USER_EMAIL"]], cwd=repo_dir, check=True)
    subprocess.run(["git", "add", str(exp_dir.relative_to(repo_dir))], cwd=repo_dir, check=True)

    status = subprocess.check_output(["git", "status", "--short"], cwd=repo_dir).decode().strip()
    if not status:
        print("No new final artifacts to commit.")
    else:
        subprocess.run(["git", "commit", "-m", f"exp: add step5 T4x2 artifacts {run_suffix}"], cwd=repo_dir, check=True)
        askpass = work_dir / "git_askpass_final_backup.sh"
        askpass.write_text(textwrap.dedent(f"""\
        #!/bin/sh
        case \"$1\" in
          *Username*) echo \"{owner}\" ;;
          *Password*) echo \"{token}\" ;;
          *) echo \"\" ;;
        esac
        """), encoding="utf-8")
        askpass.chmod(0o700)
        env = os.environ.copy()
        env["GIT_ASKPASS"] = str(askpass)
        env["GIT_TERMINAL_PROMPT"] = "0"
        subprocess.run(["git", "push", "origin", branch], cwd=repo_dir, check=True, env=env)
        print("Final artifact metadata pushed to GitHub.")
        print("GitHub folder:", f"experiments/kaggle_runs/{run_suffix}")
